<a href="https://colab.research.google.com/github/D2718281828nis/LLM_AI_agents/blob/main/lesson-2A_OrbitLM_Colab_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/D2718281828nis/LLM_AI_agents/blob/main/lesson-2A_OrbitLM_Colab_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 2A — OrbitLM и простой агент внутри Colab

В этом уроке OrbitLM запускается **непосредственно в среде выполнения ноутбука** через Hugging Face Transformers. Не нужны OpenAI-совместимый API, vLLM-сервер, URL, API-ключ или отдельная удалённая служба.

```text
Ячейка ноутбука → tokenizer.apply_chat_template → model.generate
       ↑                                               │
       └──── tool response ← Python-инструмент ← <tool_call>
```

Цели урока:

- загрузить публичную модель `ai-forever/OrbitLM-1.8B-Thinking`;
- выполнить обычный chat-запрос;
- передать схему Python-инструмента через штатный chat template;
- разобрать `<tool_call>` и выполнить цикл **Think → Action → Observation → Answer**;
- увидеть, чем четыре внутренних прохода looped LLM отличаются от внешних итераций агента.

Источники: [карточка OrbitLM](https://huggingface.co/ai-forever/OrbitLM-1.8B-Thinking) и [репозиторий модели](https://github.com/ai-forever/OrbitLM).

## 1. Подготовка среды

В Colab выберите **Runtime → Change runtime type → GPU**. Официальный пример использует BF16 и CUDA. Для GPU без поддержки BF16 ноутбук автоматически выберет FP16, однако такой режим не является основной конфигурацией из карточки модели.

При первом запуске Transformers скачает веса из публичного Hugging Face-репозитория. **Hugging Face-токен не требуется.** Параметр `trust_remote_code=True` разрешает выполнение кода архитектуры Ouro, опубликованного вместе с моделью.

In [1]:
%pip install -q "transformers>=4.56,<6" accelerate safetensors jinja2

In [2]:
import json
import re

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Проверяем доступность GPU и настраиваем девайс и тип данных соответственно
if torch.cuda.is_available():
    device = "cuda"
    MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    print("GPU обнаружен:", torch.cuda.get_device_name(0))
else:
    device = "cpu"
    MODEL_DTYPE = torch.float32
    print("ВНИМАНИЕ: GPU не найден. Модель будет запущена на CPU (это будет медленно).")
    print("Рекомендуется включить GPU в Colab: Runtime → Change runtime type → GPU")

MODEL_ID = "ai-forever/OrbitLM-1.8B-Thinking"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=MODEL_DTYPE,
    trust_remote_code=True,
    early_exit_threshold=None,
).to(device)
model.eval()

# Рекомендация из карточки модели для некоторых конфигураций CUDA
if device == "cuda":
    torch.backends.cuda.enable_cudnn_sdp(False)

print("Используемый девайс:", device)
print("dtype:", MODEL_DTYPE)
print("Внутренних проходов:", model.config.total_ut_steps)

GPU обнаружен: Tesla T4


config.json:   0%|          | 0.00/1.60k [00:00<?, ?B/s]

configuration_ouro.py:   0%|          | 0.00/13.7k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/ai-forever/OrbitLM-1.8B-Thinking:
- configuration_ouro.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.13/dist-packages/transformers/configuration_utils.py:723: FutureWarning: `rope_scaling` is deprecated and will be removed in a future version. Set `rope_parameters` instead, with `rope_type` and `rope_theta` included.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/modeling_rope_utils.py:1164: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantim

tokenizer_config.json:   0%|          | 0.00/7.57k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

... (10 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/269 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/152 [00:00<?, ?B/s]

Используемый девайс: cuda
dtype: torch.bfloat16
Внутренних проходов: 4


## 2. Функция генерации

При прямом запуске через Transformers перед шаблоном добавляется `<|endoftext|>`, как в официальном примере OrbitLM. Настройки sampling уже хранятся в `generation_config.json`: `temperature=0.6`, `top_p=0.95`, `top_k=20`.

Параметр `exit_at_step=3` означает четвёртый, то есть последний внутренний проход. Значения `0`, `1`, `2`, `3` соответствуют одному, двум, трём и четырём проходам.

In [3]:
def generate_chat(
    messages,
    tools=None,
    enable_thinking=True,
    max_new_tokens=2048,
    exit_at_step=3,
):
    prompt = "<|endoftext|>" + tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking,
    )
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False,
    ).to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            exit_at_step=exit_at_step,
        )

    generated_ids = output[0, inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(
        generated_ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )

    return (
        generated_text.replace("<|im_end|>", "")
        .replace("<|endoftext|>", "")
        .strip()
    )


def extract_answer(generated_text):
    if "</think>" in generated_text:
        return generated_text.split("</think>", 1)[1].strip()
    return generated_text.strip()

## 3. Обычный chat-запрос

Сначала проверим модель без инструментов. Для более быстрого ответа можно установить `enable_thinking=False` или уменьшить `exit_at_step`; это может изменить качество результата.

In [4]:
messages = [
    {
        "role": "user",
        "content": "Объясни в трёх предложениях, чем агент отличается от обычного LLM-запроса.",
    }
]

generated_text = generate_chat(messages, max_new_tokens=1024)
print(extract_answer(generated_text))

[transformers] The remote code model you are currently using seems to expect `cache_position`. This arg has been removed from the Transformers library, and will stop being created in `generate` even for remote code models in a future release. Please open a PR on the remote code hub repo to remove any usage of `cache_position`.


Агент отличается от обычного LLM-запроса тем, что может самостоятельно принимать решения, взаимодействовать с внешними системами и адаптироваться к условиям, тогда как обычный запрос просто предоставляет ответ на вопрос. Агент обладает автономностью и способен выполнять действия, а не только отвечать на запросы. В отличие от обычного запроса, агент может обучаться и улучшать свою работу на основе опыта.


## 4. Python-инструмент и его схема

Создадим учебного агента для расчёта индекса массы тела. Модель не выполняет Python-код сама: она возвращает `<tool_call>` с именем функции и JSON-аргументами, а ноутбук вызывает только функцию из разрешённого словаря `TOOL_FUNCTIONS`.

В прямом режиме Transformers схема передаётся в `apply_chat_template` как список описаний функций — обёртка OpenAI `type=function` не нужна. Пример демонстрирует tool calling и не предназначен для медицинских рекомендаций.

In [5]:
def calculate_bmi(weight_kg: float, height_cm: float) -> dict:
    if weight_kg <= 0 or height_cm <= 0:
        raise ValueError("Вес и рост должны быть положительными числами")

    height_m = height_cm / 100
    bmi = weight_kg / height_m**2
    return {
        "bmi": round(bmi, 1),
        "weight_kg": weight_kg,
        "height_cm": height_cm,
    }


TOOL_FUNCTIONS = {"calculate_bmi": calculate_bmi}

TOOLS = [
    {
        "name": "calculate_bmi",
        "description": "Рассчитывает индекс массы тела по весу и росту.",
        "parameters": {
            "type": "object",
            "properties": {
                "weight_kg": {
                    "type": "number",
                    "description": "Вес в килограммах",
                },
                "height_cm": {
                    "type": "number",
                    "description": "Рост в сантиметрах",
                },
            },
            "required": ["weight_kg", "height_cm"],
            "additionalProperties": False,
        },
    }
]

## 5. Минимальный TAO-агент

Функция `parse_tool_calls` извлекает JSON из `<tool_call>...</tool_call>`. Перед добавлением хода ассистента в историю блоки вызовов удаляются из `content`, поскольку chat template восстановит их из отдельного поля `tool_calls`; иначе вызовы дублировались бы.

После выполнения функции результат добавляется как сообщение `role="tool"`. При следующей генерации шаблон превратит его в `<tool_response>`, и модель сможет сформулировать итоговый ответ. Ограничение `max_model_calls` предотвращает бесконечный внешний цикл.

In [6]:
TOOL_CALL_PATTERN = re.compile(
    r"<tool_call>\s*(\{.*?\})\s*</tool_call>",
    flags=re.DOTALL,
)


def parse_tool_calls(generated_text):
    return [
        json.loads(match)
        for match in TOOL_CALL_PATTERN.findall(generated_text)
    ]


def run_orbit_agent(user_query, max_model_calls=4):
    messages = [
        {
            "role": "system",
            "content": (
                "Ты учебный ассистент. Для расчёта ИМТ всегда используй "
                "инструмент calculate_bmi. Не ставь медицинские диагнозы."
            ),
        },
        {"role": "user", "content": user_query},
    ]
    trajectory = []

    for _ in range(max_model_calls):
        generated_text = generate_chat(messages, tools=TOOLS)
        tool_calls = parse_tool_calls(generated_text)

        if not tool_calls:
            return {
                "answer": extract_answer(generated_text),
                "trajectory": trajectory,
            }

        messages.append(
            {
                "role": "assistant",
                "content": TOOL_CALL_PATTERN.sub("", generated_text).strip(),
                "tool_calls": tool_calls,
            }
        )

        for tool_call in tool_calls:
            function_name = tool_call["name"]
            arguments = tool_call["arguments"]
            if isinstance(arguments, str):
                arguments = json.loads(arguments)

            if function_name not in TOOL_FUNCTIONS:
                raise ValueError(f"Неизвестный инструмент: {function_name}")

            observation = TOOL_FUNCTIONS[function_name](**arguments)
            trajectory.append(
                {
                    "action": function_name,
                    "arguments": arguments,
                    "observation": observation,
                }
            )
            messages.append(
                {
                    "role": "tool",
                    "content": json.dumps(observation, ensure_ascii=False),
                }
            )

    raise RuntimeError("Достигнут лимит вызовов модели до получения итогового ответа")

In [7]:
result = run_orbit_agent("Мой вес 72 кг, рост 178 см. Рассчитай ИМТ.")

print("Траектория агента:")
print(json.dumps(result["trajectory"], ensure_ascii=False, indent=2))
print("\nИтоговый ответ OrbitLM:")
print(result["answer"])

Траектория агента:
[
  {
    "action": "calculate_bmi",
    "arguments": {
      "weight_kg": 72,
      "height_cm": 178
    },
    "observation": {
      "bmi": 22.7,
      "weight_kg": 72,
      "height_cm": 178
    }
  }
]

Итоговый ответ OrbitLM:
Индекс массы тела (ИМТ) рассчитан как **22.7**. Это значение находится в пределах **нормального диапазона** (18.5–24.9).  

Такой ИМТ указывает на нормальную массу тела, и при этом нет необходимости в специальных мерах по снижению веса. Однако для более точной оценки рекомендуется учитывать возраст, пол, образ жизни и другие факторы. Если хотите, могу помочь с другими показателями или разобрать, как это влияет на здоровье.


## 6. Что исследовать дальше

- Измените вес и рост и проследите новые аргументы в траектории.
- Задайте вопрос без расчётов: агент должен завершиться без вызова инструмента.
- Сравните `exit_at_step=0`, `1`, `2`, `3` по скорости, корректности аргументов и качеству ответа.
- Сравните `enable_thinking=True` и `False`, не меняя инструмент и запрос.
- Выведите сырой `generated_text` и найдите `</think>` и `<tool_call>`.

**Главное различие:** `exit_at_step` управляет повторными проходами общего стека слоёв при генерации токенов. `max_model_calls` ограничивает внешний агентный цикл «модель → инструмент → наблюдение». Это два независимых уровня вычислений.

Если Colab завершает ячейку из-за нехватки памяти, уменьшите `max_new_tokens`, используйте более короткую историю и перезапустите среду выполнения перед повторной загрузкой модели.